In [ ]:
!pip -q install "transformers>=4.40" nltk

In [ ]:
import re, math, time, random
import nltk, torch
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

# ---- same corpus + same split as the other models (R3) ----
BOOKS = ["austen-emma.txt", "austen-sense.txt", "carroll-alice.txt",
         "melville-moby_dick.txt", "chesterton-thursday.txt", "edgeworth-parents.txt"]
raw = "\n".join(gutenberg.raw(b) for b in BOOKS)

sents = []
for part in re.split(r"[.!?]+", raw.lower()):
    words = re.findall(r"[a-z']+", part)
    if len(words) >= 2:
        sents.append(words)
k = int(len(sents) * 0.9)
train_sents, test_sents = sents[:k], sents[k:]
train_text = " ".join(" ".join(s) for s in train_sents)
test_text  = " ".join(" ".join(s) for s in test_sents)
print(f"train {len(train_sents)} / test {len(test_sents)} sentences")

train 36199 / test 4023 sentences


In [ ]:
from transformers import (GPT2LMHeadModel, GPT2TokenizerFast,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = GPT2TokenizerFast.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# chunk the training text into blocks of 128 tokens
ids = tok(train_text, return_tensors="pt").input_ids[0]
BLOCK = 128
blocks = [ids[i:i+BLOCK] for i in range(0, len(ids) - BLOCK, BLOCK)]

class DS(torch.utils.data.Dataset):
    def __len__(self): return len(blocks)
    def __getitem__(self, i): return {"input_ids": blocks[i], "labels": blocks[i]}

args = TrainingArguments(output_dir="gpt2-ckpt", num_train_epochs=1,
                         per_device_train_batch_size=8, learning_rate=5e-5,
                         logging_steps=100, save_strategy="no", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=DS(),
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (756874 > 1024). Running this sequence through the model will result in indexing errors


Step,Training Loss
100,4.571329
200,4.440010
300,4.332031
400,4.320434
500,4.315709
600,4.291715
700,4.269291


TrainOutput(global_step=740, training_loss=4.359481131063926, metrics={'train_runtime': 222.3441, 'train_samples_per_second': 26.594, 'train_steps_per_second': 3.328, 'total_flos': 386254946304000.0, 'train_loss': 4.359481131063926, 'epoch': 1.0})

In [ ]:
model.eval()
test_ids = tok(test_text, return_tensors="pt").input_ids[0]
nll = 0.0
with torch.no_grad():
    for i in range(0, len(test_ids) - 1, 512):
        chunk = test_ids[i:i+512].unsqueeze(0).to(device)
        out = model(chunk, labels=chunk)
        nll += out.loss.item() * (chunk.size(1) - 1)   # loss is mean over tokens
n_words = len(test_text.split())
word_ppl = math.exp(nll / n_words)
print(f"Word-level perplexity: {word_ppl:.1f}")

Word-level perplexity: 82.7


In [ ]:
cpu_model = model.to("cpu").eval()

samples = []
for s in test_sents:
    for i in range(1, len(s)):
        samples.append((s[max(0, i-20):i], s[i]))
random.seed(0); random.shuffle(samples); samples = samples[:500]

correct = 0
t0 = time.perf_counter()
with torch.no_grad():
    for ctx_words, target in samples:
        ctx_ids = tok(" ".join(ctx_words), return_tensors="pt").input_ids
        logits = cpu_model(ctx_ids).logits[0, -1]
        top = logits.topk(3).indices.tolist()
        preds = [tok.decode([t]).strip().lower() for t in top]
        correct += target in preds
latency = (time.perf_counter() - t0) / len(samples) * 1000
print(f"Top-3 next-word accuracy: {correct/len(samples):.4f}")
print(f"Latency: {latency:.1f} ms/suggestion (CPU)")

Top-3 next-word accuracy: 0.3560
Latency: 124.6 ms/suggestion (CPU)


In [ ]:
from huggingface_hub import login
login()   # paste your WRITE token

In [ ]:
repo = "tia88na/gpt2-keyboard"
model.push_to_hub(repo)
tok.push_to_hub(repo)
print("Model URL:", "https://huggingface.co/" + repo)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kvpn3ly/model.safetensors:   0%|          |  549kB /  498MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Model URL: https://huggingface.co/tia88na/gpt2-keyboard
